# NOMOPHOBIA CPU S3 + Submission

**Purpose:** run production-scale repeated iteration tuning, the authoritative 3-seed × 5-fold S3 campaign, mechanical promotion, and final validated seed-bag submission in one CPU notebook.

### Required Kaggle input
- Official competition input containing `train.csv`, `test.csv`, and `sample_submission.csv`.

### Recommended Kaggle input
- Saved output from **Notebook 1: CPU Research Suite**. This is used for a preflight decision summary only; S3 remains mechanically independent.

### Notebook settings
- **Accelerator:** None (CPU)
- **Internet:** **ON** for GitHub bootstrap, or OFF if `nomophobia-source.zip` is attached.
- **Persistence:** recommended while interactively testing. The campaign is restart-safe inside the same `/kaggle/working` tree and reuses completed tuning/seed stages only when frozen iteration counts match.

This notebook is intentionally expensive. Inspect the notebook editor's current resource limits before launching a full saved run.

In [ ]:
from pathlib import Path
import os, json, multiprocessing, shutil, zipfile, subprocess

REPO_REF = "main"
THREADS = max(1, multiprocessing.cpu_count())
DATA_DIR = Path("/kaggle/input/playground-series-s6e8")
OUT_ROOT = Path("/kaggle/working/nomophobia_cpu_s3")
REPO_DIR = Path("/kaggle/working/nomophobia")

TUNE_ROWS = 628_000
MAX_ESTIMATORS = 4_000
PATIENCE = 200
TUNE_REPEATS = 3
BOOTSTRAP = 1_200
SEEDS = [20260816, 20260817, 20260818]

assert (DATA_DIR / "train.csv").exists(), "Attach the Playground Series S6E8 competition input."
print("CPU threads:", THREADS)


## Read Notebook 1 decision, if attached

A red source-safety stop should be investigated before spending full S3 compute. New experimental candidates do **not** replace the baseline here; S3 is still valuable because it freezes the reference model they must beat.

In [ ]:
research_decision = None
for p in Path("/kaggle/input").rglob("cpu_research_decision.json"):
    try:
        research_decision = json.loads(p.read_text())
        print("Found research decision:", p)
        break
    except Exception:
        pass

if research_decision:
    print(json.dumps({
        "recommended_next_step": research_decision.get("recommended_next_step"),
        "frequency_safety": research_decision.get("frequency_safety"),
        "capacity_route": research_decision.get("capacity_route"),
        "geometry_advanced_arms": research_decision.get("geometry_advanced_arms"),
        "source_advanced_weights": research_decision.get("source_advanced_weights"),
    }, indent=2))
    if research_decision.get("recommended_next_step") == "STOP_FREQUENCY_EXPANSION_AUDIT_SOURCE_SHIFT":
        raise RuntimeError("Notebook 1 triggered the frequency source-safety stop. Audit before S3.")
else:
    print("Notebook 1 output not attached; proceeding with the audited baseline configuration.")


## Bootstrap repository

In [ ]:
def find_offline_repo():
    for candidate in Path("/kaggle/input").rglob("pyproject.toml"):
        parent = candidate.parent
        if (parent / "src" / "s6e8").exists():
            return parent
    for z in Path("/kaggle/input").rglob("nomophobia-source.zip"):
        target = Path("/kaggle/working/nomophobia_offline")
        if target.exists():
            shutil.rmtree(target)
        target.mkdir(parents=True)
        with zipfile.ZipFile(z) as f:
            f.extractall(target)
        for candidate in target.rglob("pyproject.toml"):
            if (candidate.parent / "src" / "s6e8").exists():
                return candidate.parent
    return None

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
try:
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", REPO_REF,
        "https://github.com/sidhulyalkar/nomophobia.git", str(REPO_DIR)
    ], check=True)
except Exception:
    offline = find_offline_repo()
    if offline is None:
        raise RuntimeError("Enable Internet or attach nomophobia-source.zip.")
    shutil.copytree(offline, REPO_DIR)

subprocess.run([os.sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR), "--no-deps"], check=True)
subprocess.run(["nomophobia", "validate", "--data-dir", str(DATA_DIR)], check=True)


## Dry-run the authoritative campaign plan

This prints the frozen orchestration parameters before any expensive fit.

In [ ]:
print(json.dumps({
    "tune_rows": TUNE_ROWS,
    "max_estimators": MAX_ESTIMATORS,
    "patience": PATIENCE,
    "tune_repeats": TUNE_REPEATS,
    "seeds": SEEDS,
    "folds_per_seed": 5,
    "experts": ["lgb_combined63", "lgb_raw63"],
    "device": "cpu",
}, indent=2))


## Run resumable tuning + S3 + submission

The wrapper:
- tunes combined/raw independently on repeated production-scale holdouts,
- stops before S3 if either tuning repeat hits the estimator ceiling,
- reuses completed S3 seed directories only when their iteration overrides match,
- reruns promotion aggregation,
- materializes `submission_s3.csv` only when the mechanical route freezes a promoted backbone,
- creates one ZIP containing all authoritative artifacts.

In [ ]:
cmd = [
    os.sys.executable,
    str(REPO_DIR / "scripts" / "run_cpu_s3_campaign.py"),
    "--data-dir", str(DATA_DIR),
    "--out-root", str(OUT_ROOT),
    "--tune-rows", str(TUNE_ROWS),
    "--max-estimators", str(MAX_ESTIMATORS),
    "--patience", str(PATIENCE),
    "--tune-repeats", str(TUNE_REPEATS),
    "--bootstrap", str(BOOTSTRAP),
    "--seeds", *[str(x) for x in SEEDS],
    "--threads", str(THREADS),
]
print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)


## Final S3 result and submission artifact

In [ ]:
summary = json.loads((OUT_ROOT / "cpu_s3_summary.json").read_text())
print(json.dumps({
    "status": summary.get("status"),
    "selected_iterations": summary.get("selected_iterations"),
    "resolution": summary.get("resolution"),
    "submission": summary.get("submission"),
    "elapsed_hours": round(summary.get("elapsed_seconds", 0) / 3600, 2),
}, indent=2))

submission = OUT_ROOT / "submission_s3.csv"
if submission.exists():
    print("\nKaggle submission ready:", submission)
else:
    print("\nNo submission was materialized because the S3 promotion route did not freeze a deployable candidate.")

print("\nFiles to preserve:")
for p in sorted(Path("/kaggle/working").glob("nomophobia_cpu_s3*")):
    print(" ", p)


### After the run

Use **Save Version** with output files enabled. If `submission_s3.csv` exists, upload that exact file to Kaggle. Preserve `nomophobia_cpu_s3_artifacts.zip` and `cpu_s3_summary.json` so future model candidates can be paired against the frozen S3 backbone rather than rerunning it unnecessarily.